# 08 · Recommendation

**Stage 8 of 8** · ACCESS → WRANGLE → EXPLORE → VISUALIZE → ENGINEER → MODEL → EVALUATE → RECOMMEND

> **Project premise:** if someone has had cancer and was treated, **will it come back?** And for every factor linked to it coming back: **correlation or causation?**
>
> All data here is public: a k-anonymous feature table (every combination of site group, region, sex and diagnosis period is shared by ≥ 3 patients), synthetic patients, and aggregate reports. The raw database and clinical notes stay private.

**Goal of this stage:** answer the premise: correlation or causation. Then turn the findings into actions and state the limits.

**Business question:** Will the cancer come back, and which links can we treat as cause and effect?

In [1]:
%matplotlib inline
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
PUBLIC = ROOT / "data" / "public" / "patients_features_public.csv"
SYNTH = ROOT / "data" / "synthetic" / "synthetic_patients.csv"
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8984"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.axisbelow": True, "figure.dpi": 110})
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
df = pd.read_csv(PUBLIC)

## The finding (from the full analysis, `reports/tables/correlation_vs_causation.csv`)

In [2]:
t = pd.read_csv(ROOT / "reports" / "tables" / "correlation_vs_causation.csv")
t[["factor", "population", "relapse_with", "relapse_without", "crude_HR", "adjusted_HR", "adjusted_CI",
   "E_value", "verdict"]]

,factor,population,relapse_with,relapse_without,crude_HR,adjusted_HR,adjusted_CI,E_value,verdict
0,Breast cancer (vs other sites),all sites,18/94 (19%),12/44 (27%),0.33,0.33,0.16-0.70,3.65,"Risk marker: real association, but not somethi..."
1,Hormone-receptor positive (breast),breast only,4/24 (17%),14/70 (20%),0.98,0.98,0.32-3.00,1.13,No clear link (too few patients to tell)
2,Triple-negative (breast),breast only,3/16 (19%),15/78 (19%),1.24,1.30,0.33-5.12,1.69,No clear link (too few patients to tell)
3,Grade 3 tumour,all sites,4/20 (20%),26/118 (22%),0.68,0.91,0.31-2.66,1.33,No clear link (too few patients to tell)
4,Surgery,all sites,10/43 (23%),20/95 (21%),0.78,0.78,0.36-1.67,1.66,No clear link (too few patients to tell)
5,Radiotherapy,all sites,15/83 (18%),15/55 (27%),0.54,0.55,0.26-1.16,2.40,No clear link (too few patients to tell)
6,Hormonal therapy,breast only,12/69 (17%),6/25 (24%),0.38,0.36,0.12-1.04,3.45,No clear link (too few patients to tell)
7,Neoadjuvant chemo (large tumour at start),all sites,3/13 (23%),27/125 (22%),0.88,1.18,0.35-3.94,1.49,No clear link (too few patients to tell)
8,Treated abroad,all sites,3/10 (30%),27/128 (21%),1.31,1.38,0.42-4.48,1.80,No clear link (too few patients to tell)
9,Missed follow-up / delayed care,all sites,1/8 (12%),29/130 (22%),0.51,0.35,0.05-2.37,3.55,No clear link (too few patients to tell)


### How each verdict was reached
For a link to count as **causal**, we asked four questions:
1. **Correlation:** is there a link at all? (crude hazard ratio)
2. **Adjustment:** does it survive adjusting for measured confounders? (Cox model)
3. **Robustness:** how strong would a hidden confounder, such as *stage*, have to be to explain it away? (E-value)
4. **Outside evidence:** do clinical trials agree?

Demonstrated for hormonal therapy (breast cancer only) on the public data:

In [3]:
import analysis as A
%matplotlib inline
c = A.cohort(df)
b = c[c.breast == 1]
crude = A.cox_hr(b, "hormonal_therapy", [])
adj = A.cox_hr(b, "hormonal_therapy", ["hr_pos", "surgery", "radiotherapy"])
print(f"crude HR {crude[0]:.2f} ({crude[1]:.2f}–{crude[2]:.2f})")
print(f"adjusted HR {adj[0]:.2f} ({adj[1]:.2f}–{adj[2]:.2f})")
print("E-value (point, CI):", A.e_value(*adj[:3]))
print("relapses needed to detect HR 0.5 with 80% power:", A.events_needed(0.5), "— we have", int(c.relapse.sum()))

crude HR 0.38 (0.14–1.01)
adjusted HR 0.36 (0.12–1.04)
E-value (point, CI): (3.45, 1.0)
relapses needed to detect HR 0.5 with 80% power: 66 — we have 30


## The answer to the premise
| | Answer |
|---|---|
| **Will it come back?** | For **about 1 in 5** treated patients (22%). Relapse-free ≈ 87% at 2 yrs, 78% at 5 yrs. |
| **Clear link** | **Cancer type**: breast cancer came back less than other cancers. A real association, but a *risk marker*, not a modifiable cause. |
| **Correlation, consistent with causation, not proven** | **Hormonal therapy** and **radiotherapy**: both point to fewer relapses, the same direction as clinical trials, but their confidence intervals include "no effect", and unrecorded stage could explain them. |
| **No clear link** | Surgery, grade, triple-negative, region, treated abroad, missed follow-up |

## Recommendations
1. **Record the variables that make causation answerable**: stage, age, ER/PR/HER2, date of relapse, date of death,
   as structured fields. Receptor status is missing for 68% of breast patients, and stage and age for everyone.
2. **Plan follow-up by cancer type.** Non-breast cancers came back sooner and more often. That's the link this data supports.
3. **Don't use the risk model for care decisions.** Its cross-validated C-index of about 0.63 is weak.
4. **Re-run once ~66+ relapses are recorded** with stage and age, and validate on new patients.

## Limitations
- One clinic, 335 patients, 2020–21; 30 relapses.
- Only patients alive and attending are included (survivor bias), so 22% likely **understates** relapse.
- Facts come from free-text notes via rules, so some will be wrong.
- Reverse causation: treatment lists can change *because* of a relapse.

*These are associations from one clinic, not medical advice.*

## So what?

**Correlation: yes** (cancer type clearly, treatments weakly). **Causation: not provable with this data.** The most
valuable output of the project is knowing exactly which missing variables would turn this into a causal answer.